# All-Lecture Ingestion — MIT 18.06 Linear Algebra

Dynamic ingestion pipeline for multiple lectures.

Change only:

```python
LECTURE_START = 1
LECTURE_END = 34
```

Pipeline:

`video → FFmpeg audio → faster-whisper STT → timestamped JSON/TXT → lecture metadata`


# Step 0: Setup & Project Configuration


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/AI_Engineering_Final_Project"
)

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

VIDEO_DIR = RAW_DIR / "videos"
AUDIO_DIR = RAW_DIR / "audio"
TEXT_DIR = PROCESSED_DIR / "transcripts"
METADATA_DIR = PROCESSED_DIR / "metadata"

for folder in [VIDEO_DIR, AUDIO_DIR, TEXT_DIR, METADATA_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project folders ready.")


Project folders ready.


In [ ]:
LECTURE_START = 1
LECTURE_END = 34

COURSE_ID = "MIT_18_06"
COURSE_NAME = "Linear Algebra"
INSTRUCTOR = "Gilbert Strang"
SOURCE = "MIT OpenCourseWare"

COURSE_URL = (
    "https://ocw.mit.edu/courses/"
    "18-06-linear-algebra-spring-2010/"
)

LECTURE_LIST_URL = (
    COURSE_URL
    + "video_galleries/video-lectures/"
)

VIDEO_BASE_URL = (
    "https://archive.org/download/"
    "MIT18.06S05_MP4"
)

print(f"Lectures selected: {LECTURE_START}–{LECTURE_END}")


Lectures selected: 1–34


In [ ]:
!pip install -q beautifulsoup4


In [ ]:
# Fetch official lecture titles dynamically from MIT OpenCourseWare.
# No 34-item hard-coded title dictionary is maintained in this notebook.

import re
import requests
from html import unescape
from bs4 import BeautifulSoup


def fetch_official_lecture_titles(url, lecture_start=1, lecture_end=34):
    response = requests.get(url, timeout=30)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    titles = {}

    # MIT OCW link text follows:
    # "Lecture 1: The geometry of linear equations"
    # ...
    # "Lecture 24b: Quiz 2 review"
    #
    # The regex intentionally accepts only integer lecture numbers,
    # so supplemental "Lecture 24b" is not mapped onto Lecture 24.
    pattern = re.compile(r"^Lecture\s+(\d+):\s*(.+)$", re.IGNORECASE)

    for link in soup.find_all("a"):
        text = " ".join(link.get_text(" ", strip=True).split())
        match = pattern.match(text)

        if not match:
            continue

        lecture_number = int(match.group(1))
        lecture_title = unescape(match.group(2)).strip()

        if lecture_start <= lecture_number <= lecture_end:
            titles[lecture_number] = lecture_title

    expected = set(range(lecture_start, lecture_end + 1))
    found = set(titles)
    missing = sorted(expected - found)

    if missing:
        raise RuntimeError(
            "Could not read official MIT OCW titles for lectures: "
            + ", ".join(map(str, missing))
        )

    return {
        number: titles[number]
        for number in range(lecture_start, lecture_end + 1)
    }


OFFICIAL_LECTURE_TITLES = fetch_official_lecture_titles(
    LECTURE_LIST_URL,
    LECTURE_START,
    LECTURE_END,
)

print(f"Official lecture titles loaded: {len(OFFICIAL_LECTURE_TITLES)}")
print("Source:", LECTURE_LIST_URL)

for number in range(LECTURE_START, min(LECTURE_START + 4, LECTURE_END + 1)):
    print(f"Lecture {number}: {OFFICIAL_LECTURE_TITLES[number]}")


Official lecture titles loaded: 34
Source: https://ocw.mit.edu/courses/18-06-linear-algebra-spring-2010/video_galleries/video-lectures/
Lecture 1: The geometry of linear equations
Lecture 2: Elimination with matrices
Lecture 3: Multiplication and inverse matrices
Lecture 4: Factorization into A = LU


In [ ]:
def build_lecture_config(lecture_number):
    lecture_id = f"L{lecture_number:02d}"
    lecture_slug = f"lecture_{lecture_number:02d}"

    return {
        "course_id": COURSE_ID,
        "course_name": COURSE_NAME,
        "lecture_id": lecture_id,
        "lecture_number": lecture_number,
        "lecture_title": OFFICIAL_LECTURE_TITLES[lecture_number],
        "instructor": INSTRUCTOR,
        "source": SOURCE,
        "course_url": COURSE_URL,
        "lecture_title_source_url": LECTURE_LIST_URL,

        # Archive.org filenames follow 01.mp4, 02.mp4, ...
        "video_url": f"{VIDEO_BASE_URL}/{lecture_number:02d}.mp4",

        "video_path": VIDEO_DIR / f"{lecture_slug}.mp4",
        "audio_path": AUDIO_DIR / f"{lecture_slug}.wav",
        "stt_json_path": TEXT_DIR / f"{lecture_slug}_stt.json",
        "stt_txt_path": TEXT_DIR / f"{lecture_slug}_stt.txt",
        "metadata_path": METADATA_DIR / f"{lecture_slug}_metadata.json",
    }


LECTURES = [
    build_lecture_config(i)
    for i in range(LECTURE_START, LECTURE_END + 1)
]

LECTURES[:2]


# Step 1: Data Acquisition


In [ ]:
import requests

def download_file(url, output_path):
    output_path = Path(output_path)

    if output_path.exists():
        print(f"Already exists: {output_path.name}")
        return output_path

    print(f"Downloading: {output_path.name}")

    with requests.get(url, stream=True, timeout=120) as response:
        response.raise_for_status()

        with open(output_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)

    print(f"Saved: {output_path}")
    return output_path


In [ ]:
import subprocess
import shutil

assert shutil.which("ffmpeg"), "FFmpeg is not available in this runtime."

def extract_audio(video_path, audio_path):
    video_path = Path(video_path)
    audio_path = Path(audio_path)

    if audio_path.exists():
        print(f"Already exists: {audio_path.name}")
        return audio_path

    command = [
        "ffmpeg",
        "-y",
        "-i", str(video_path),
        "-vn",
        "-ac", "1",
        "-ar", "16000",
        "-acodec", "pcm_s16le",
        str(audio_path),
    ]

    subprocess.run(
        command,
        check=True,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

    print(f"Audio created: {audio_path.name}")
    return audio_path


# Step 2: Speech-to-Text (faster-whisper)


In [ ]:
!pip install -q faster-whisper


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 62.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 57.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.6/39.6 MB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 84.3 MB/s eta 0:00:00


In [ ]:
import torch
from faster_whisper import WhisperModel

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

if DEVICE == "cuda":
    whisper_model = WhisperModel(
        "small",
        device="cuda",
        compute_type="float16"
    )
else:
    whisper_model = WhisperModel(
        "small",
        device="cpu",
        compute_type="int8"
    )

print("Device:", DEVICE)
print("Whisper model loaded.")


Device: cuda
Whisper model loaded.


In [ ]:
import json

def transcribe_lecture(lecture):
    stt_json_path = lecture["stt_json_path"]

    # Resumable: reuse existing STT output
    if stt_json_path.exists():
        print(f"Already transcribed: {lecture['lecture_id']}")
        with open(stt_json_path, "r", encoding="utf-8") as f:
            return json.load(f)

    segments, info = whisper_model.transcribe(
        str(lecture["audio_path"]),
        beam_size=5,
        language="en"
    )

    whisper_segments = []

    for i, segment in enumerate(segments):
        whisper_segments.append({
            "segment_id": i,
            "start_seconds": round(segment.start, 2),
            "end_seconds": round(segment.end, 2),
            "text": segment.text.strip(),
            "course_id": lecture["course_id"],
            "lecture_id": lecture["lecture_id"],
            "lecture_number": lecture["lecture_number"],
            "lecture_title": lecture["lecture_title"],
            "video_url": lecture["video_url"],
            "source_type": "stt_transcript",
        })

    print(
        f"{lecture['lecture_id']} | "
        f"{len(whisper_segments)} segments | "
        f"language={info.language}"
    )

    return whisper_segments


In [ ]:
def save_transcript(lecture, whisper_segments):
    with open(
        lecture["stt_json_path"],
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            whisper_segments,
            f,
            indent=2,
            ensure_ascii=False
        )

    lines = [
        f"[{s['start_seconds']:.2f} - {s['end_seconds']:.2f}] {s['text']}"
        for s in whisper_segments
    ]

    lecture["stt_txt_path"].write_text(
        "\n".join(lines),
        encoding="utf-8"
    )


# Step 3: Lecture Metadata


In [ ]:
def get_video_duration(video_path):
    probe = subprocess.run(
        [
            "ffprobe",
            "-v", "error",
            "-show_entries", "format=duration",
            "-of", "default=noprint_wrappers=1:nokey=1",
            str(video_path),
        ],
        capture_output=True,
        text=True,
        check=True,
    )

    return float(probe.stdout.strip())


def save_lecture_metadata(lecture, segment_count):
    duration_seconds = get_video_duration(lecture["video_path"])

    metadata = {
        "course_id": lecture["course_id"],
        "course_name": lecture["course_name"],
        "lecture_id": lecture["lecture_id"],
        "lecture_number": lecture["lecture_number"],
        "lecture_title": lecture["lecture_title"],
        "lecture_title_source_url": lecture["lecture_title_source_url"],
        "instructor": lecture["instructor"],
        "source": lecture["source"],
        "course_url": lecture["course_url"],
        "video_url": lecture["video_url"],
        "video_path": str(lecture["video_path"]),
        "audio_path": str(lecture["audio_path"]),
        "stt_transcript_json": str(lecture["stt_json_path"]),
        "stt_transcript_txt": str(lecture["stt_txt_path"]),
        "duration_seconds": round(duration_seconds, 2),
        "stt_segment_count": segment_count,
    }

    with open(
        lecture["metadata_path"],
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            metadata,
            f,
            indent=2,
            ensure_ascii=False
        )

    return metadata


# Step 4: Run Dynamic Ingestion


In [ ]:
def process_lecture(lecture):
    print()
    print("=" * 60)
    print(
        f"PROCESSING {lecture['lecture_id']} "
        f"(Lecture {lecture['lecture_number']})"
    )
    print("=" * 60)

    download_file(
        lecture["video_url"],
        lecture["video_path"]
    )

    extract_audio(
        lecture["video_path"],
        lecture["audio_path"]
    )

    whisper_segments = transcribe_lecture(lecture)

    save_transcript(
        lecture,
        whisper_segments
    )

    metadata = save_lecture_metadata(
        lecture,
        len(whisper_segments)
    )

    return {
        "lecture": lecture,
        "segments": whisper_segments,
        "metadata": metadata,
    }


In [ ]:
results = []

for lecture in LECTURES:
    result = process_lecture(lecture)
    results.append(result)

print()
print(f"Finished processing {len(results)} lectures.")



PROCESSING L01 (Lecture 1)
Downloading: lecture_01.mp4
Saved: /content/drive/MyDrive/AI_Engineering_Final_Project/data/raw/videos/lecture_01.mp4
Audio created: lecture_01.wav
L01 | 280 segments | language=en

PROCESSING L02 (Lecture 2)
Downloading: lecture_02.mp4
Saved: /content/drive/MyDrive/AI_Engineering_Final_Project/data/raw/videos/lecture_02.mp4
Audio created: lecture_02.wav
L02 | 733 segments | language=en

PROCESSING L03 (Lecture 3)
Downloading: lecture_03.mp4
Saved: /content/drive/MyDrive/AI_Engineering_Final_Project/data/raw/videos/lecture_03.mp4
Audio created: lecture_03.wav
L03 | 661 segments | language=en

PROCESSING L04 (Lecture 4)
Downloading: lecture_04.mp4
Saved: /content/drive/MyDrive/AI_Engineering_Final_Project/data/raw/videos/lecture_04.mp4
Audio created: lecture_04.wav
L04 | 233 segments | language=en

PROCESSING L05 (Lecture 5)
Downloading: lecture_05.mp4
Saved: /content/drive/MyDrive/AI_Engineering_Final_Project/data/raw/videos/lecture_05.mp4
Audio created: lec

# Step 5: Validation


In [ ]:
all_ok = True

for result in results:
    lecture = result["lecture"]
    segments = result["segments"]

    checks = {
        "Video": lecture["video_path"].exists(),
        "Audio": lecture["audio_path"].exists(),
        "STT JSON": lecture["stt_json_path"].exists(),
        "STT TXT": lecture["stt_txt_path"].exists(),
        "Metadata": lecture["metadata_path"].exists(),
        "Official title": lecture["lecture_title"] == OFFICIAL_LECTURE_TITLES[lecture["lecture_number"]],
        "Timestamped segments": len(segments) > 0,
    }

    lecture_ok = all(checks.values())
    all_ok = all_ok and lecture_ok

    symbol = "✅" if lecture_ok else "❌"
    print(
        f"{symbol} {lecture['lecture_id']} | "
        f"{len(segments)} STT segments"
    )

print()
print(
    "READY FOR MULTI-LECTURE RAG ✅"
    if all_ok
    else "FIX FAILED INGESTION CHECKS ❌"
)


✅ L01 | 280 STT segments
✅ L02 | 733 STT segments
✅ L03 | 661 STT segments
✅ L04 | 233 STT segments
✅ L05 | 682 STT segments
✅ L06 | 658 STT segments
✅ L07 | 647 STT segments
✅ L08 | 680 STT segments
✅ L09 | 731 STT segments
✅ L10 | 714 STT segments
✅ L11 | 437 STT segments
✅ L12 | 722 STT segments
✅ L13 | 601 STT segments
✅ L14 | 647 STT segments
✅ L15 | 692 STT segments
✅ L16 | 652 STT segments
✅ L17 | 726 STT segments
✅ L18 | 725 STT segments
✅ L19 | 719 STT segments
✅ L20 | 707 STT segments
✅ L21 | 583 STT segments
✅ L22 | 355 STT segments
✅ L23 | 607 STT segments
✅ L24 | 752 STT segments
✅ L25 | 652 STT segments
✅ L26 | 684 STT segments
✅ L27 | 753 STT segments
✅ L28 | 686 STT segments
✅ L29 | 594 STT segments
✅ L30 | 714 STT segments
✅ L31 | 703 STT segments
✅ L32 | 645 STT segments
✅ L33 | 625 STT segments
✅ L34 | 628 STT segments

READY FOR MULTI-LECTURE RAG ✅


In [ ]:
# Quick sanity check

for result in results:
    lecture = result["lecture"]
    segments = result["segments"]

    middle = segments[len(segments) // 2]

    print()
    print(lecture["lecture_id"])
    print("Start:", middle["start_seconds"])
    print("Text:", middle["text"][:250])



L01
Start: 1202.56
Text: so z actually can be anything. Again, it's going to be another plane. Each row in a three by three

L02
Start: 1340.56
Text: left by some, let's say, 1, 2, 7.

L03
Start: 1575.82
Text: I could use what I'm saying here.

L04
Start: 1007.76
Text: product of inverses. Now you still can ask, why is this guy preferring inverses? And let

L05
Start: 1447.2
Text: And you know the picture that goes with it.

L06
Start: 1483.36
Text: that's really why we're interested in this column space,

L07
Start: 1211.4
Text: I didn't comment on that, but I should have.

L08
Start: 1465.16
Text: and then find those special solutions.

L09
Start: 1644.72
Text: We're looking for a basis for R3.

L10
Start: 1501.48
Text: of this matrix are identical.

L11
Start: 1819.16
Text: If I have a couple of v and a w and I add them,

L12
Start: 1398.4
Text: because they'll have r pivots that has rank r.

L13
Start: 1446.36
Text: You have to call up your brother or something and ask him for the